# CNN + GloVe (starter) — Jigsaw toxic comments

**Preprocessing:** `preprocess_for_cnn` (word indices, train-only vocab).

**Quick iteration:** The next code cell sets `QUICK_ITERATION = True` (small data, short sequences, small vocab) to verify the pipeline end-to-end in seconds or minutes. Set **`QUICK_ITERATION = False`** before real experiments.

**Next steps for your proposal:** Download GloVe (e.g. `glove.6B.100d.txt`), implement `build_glove_weight_matrix(vocab, path, dim)` aligned to `data.vocab`, pass `embedding.weight.data.copy_(...)`. Until then this notebook uses **trainable** random embeddings so the pipeline runs.

**Metrics (course proposal):** micro/macro F1, precision/recall/ROC-AUC per label, confusion matrices, training time, parameter count.

In [1]:
from pathlib import Path
import sys

# Repo root (contains preprocessing/)
ROOT = Path.cwd().resolve()
for c in (ROOT, ROOT.parent):
    if (c / "preprocessing" / "text_preprocessing.py").exists():
        ROOT = c
        break
sys.path.insert(0, str(ROOT))
# notebooks/ for metrics_helpers
sys.path.insert(0, str(ROOT / "notebooks"))

In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset

from preprocessing.text_preprocessing import LABEL_COLUMNS
from metrics_helpers import multilabel_evaluation_report, per_label_confusion_matrices, torch_parameter_count

def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = pick_device()
print("Device:", DEVICE)
RNG = np.random.default_rng(42)
torch.manual_seed(42)

from preprocessing.text_preprocessing import preprocess_for_cnn

# --- Quick iteration (smoke test). Set False for full training. ---
QUICK_ITERATION = True

if QUICK_ITERATION:
    _train_n, _val_n = 2048, 512
    MAX_LEN = 64
    BATCH_SIZE = 128
    EPOCHS = 1
    MAX_VOCAB = 3000
    MIN_FREQ = 1
else:
    _train_n, _val_n = None, None
    MAX_LEN = 256
    BATCH_SIZE = 64
    EPOCHS = 1
    MAX_VOCAB = 50_000
    MIN_FREQ = 2

LR = 1e-3

data = preprocess_for_cnn(
    max_len=MAX_LEN,
    validation_fraction=0.1,
    random_state=42,
    min_freq=MIN_FREQ,
    max_vocab=MAX_VOCAB,
    max_train_samples=_train_n,
    max_val_samples=_val_n,
)
vocab_size = len(data.vocab)
embed_dim = 100
num_labels = len(LABEL_COLUMNS)
print("QUICK_ITERATION:", QUICK_ITERATION, "| Train/val:", data.X_train.shape, data.X_val.shape)

Device: mps
QUICK_ITERATION: True | Train/val: (2048, 64) (512, 64)


In [3]:
class TextCNN(nn.Module):
    # Kim-style multi-channel CNN over embeddings

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        num_labels: int,
        padding_idx: int = 0,
        kernel_sizes: tuple[int, ...] = (3, 4, 5),
        num_filters: int = 100,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes])
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_labels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e = self.embedding(x).transpose(1, 2)
        pools = []
        for conv in self.convs:
            h = torch.relu(conv(e))
            pools.append(h.max(dim=2).values)
        h = torch.cat(pools, dim=1)
        return self.fc(self.dropout(h))


model = TextCNN(vocab_size, embed_dim, num_labels, padding_idx=0).to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

train_ds = TensorDataset(
    torch.tensor(data.X_train, dtype=torch.long),
    torch.tensor(data.y_train, dtype=torch.float32),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

t0 = time.perf_counter()
model.train()
for epoch in range(EPOCHS):
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
train_seconds = time.perf_counter() - t0
print(f"Training wall time ({EPOCHS} epoch(s)): {train_seconds:.1f} s")

Training wall time (1 epoch(s)): 2.8 s


In [4]:
def predict_logits(model, X_np: np.ndarray, batch_size: int = 512) -> np.ndarray:
    model.eval()
    out = []
    x = torch.tensor(X_np, dtype=torch.long)
    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            logits = model(x[i : i + batch_size].to(DEVICE))
            out.append(logits.cpu().numpy())
    return np.concatenate(out, axis=0)


logits_val = predict_logits(model, data.X_val)
prob_val = torch.sigmoid(torch.from_numpy(logits_val)).numpy()
pred_val = (prob_val >= 0.5).astype(np.int64)

per_label_df, summary = multilabel_evaluation_report(
    data.y_val, pred_val, prob_val, LABEL_COLUMNS
)
print("Micro / macro F1:", summary)
display(per_label_df)
confs = per_label_confusion_matrices(data.y_val, pred_val, LABEL_COLUMNS)
for name, m in confs.items():
    print(name, "\n", m)

print()
print("--- Proposal summary (record for report / comparison) ---")
print(f"  device: {DEVICE}")
print(f"  training_time_s: {train_seconds:.2f}")
print(f"  num_parameters: {torch_parameter_count(model)}")
if summary["f1_macro"] == 0.0 and summary["f1_micro"] == 0.0:
    print(
        "  Note: F1 can be 0 if every predicted label is 0 at threshold 0.5 (common after 1 smoke epoch)."
        " ROC-AUC may still be > 0.5. Raise EPOCHS or set QUICK_ITERATION=False for real training."
    )

Micro / macro F1: {'f1_micro': 0.0, 'f1_macro': 0.0}


,label,precision,recall,f1,roc_auc
0,toxic,0.0,0.0,0.0,0.693217
1,severe_toxic,0.0,0.0,0.0,0.413386
2,obscene,0.0,0.0,0.0,0.661894
3,threat,0.0,0.0,0.0,0.585127
4,insult,0.0,0.0,0.0,0.663240
5,identity_hate,0.0,0.0,0.0,0.524063


toxic 
 [[457   0]
 [ 55   0]]
severe_toxic 
 [[508   0]
 [  4   0]]
obscene 
 [[485   0]
 [ 27   0]]
threat 
 [[511   0]
 [  1   0]]
insult 
 [[483   0]
 [ 29   0]]
identity_hate 
 [[507   0]
 [  5   0]]

--- Proposal summary (record for report / comparison) ---
  device: mps
  training_time_s: 2.82
  num_parameters: 422106
  Note: F1 can be 0 if every predicted label is 0 at threshold 0.5 (common after 1 smoke epoch). ROC-AUC may still be > 0.5. Raise EPOCHS or set QUICK_ITERATION=False for real training.


## Proposal checklist (report)

- Compare **training time** and **parameter count** across the four model notebooks.
- **Per-label threshold tuning** on validation (default here: 0.5) for rare classes such as `threat`.
- **Model size on disk**: save `state_dict` or full checkpoint and report file size in MB.
- Optional: **macro** vs **micro** trade-offs given imbalance (already reported above).